# Derivatives and Gradients

Companion notebook for the [Derivatives and Gradients](https://ml-viz-ruby.vercel.app/courses/calculus-for-ml/01-derivatives-and-gradients) lesson.

We'll visualize derivatives, compute gradients analytically and numerically, and implement gradient descent.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive. Changes to this view are not saved.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('dark_background')
plt.rcParams.update({
    'figure.facecolor': '#0f1117', 'axes.facecolor': '#1a1d27',
    'axes.edgecolor': '#30344a', 'text.color': '#e2e8f0',
    'axes.labelcolor': '#e2e8f0', 'xtick.color': '#94a3b8', 'ytick.color': '#94a3b8',
})

## Intuition — derivatives are how models learn

A **derivative** answers one question: *if I nudge the input a little, how much does the
output move?* That single number — a slope, a sensitivity — is the entire basis of
training. For a function of many inputs, the **gradient** collects one partial
derivative per input into a vector that points in the direction of steepest *increase*;
gradient descent just walks the **opposite** way to shrink a loss. This notebook builds
gradients three ways — by hand, by finite differences, and by automatic
differentiation — and shows they all agree.

## 1. Gradients from scratch — analytical vs finite differences

Two ways to get a derivative without a library. **Analytically**, you apply calculus
rules by hand: for `g(x) = x₀² + 2x₁² + x₀x₁` the partials are `∂g/∂x₀ = 2x₀ + x₁` and
`∂g/∂x₁ = 4x₁ + x₀`. **Numerically**, you approximate each partial with a *central
difference* — nudge one coordinate by `±ε` and divide by `2ε`. The cell computes both
and reports the gap, which is how gradient-checking code catches bugs in hand-derived
gradients.

In [ ]:
def g(x):
    """A function of multiple variables."""
    return x[0]**2 + 2*x[1]**2 + x[0]*x[1]

def grad_g(x):
    """Analytical gradient."""
    return np.array([2*x[0] + x[1], 4*x[1] + x[0]])

def numerical_grad(f, x, eps=1e-5):
    """Central difference approximation."""
    grad = np.zeros_like(x)
    for i in range(len(x)):
        x_plus = x.copy(); x_plus[i] += eps
        x_minus = x.copy(); x_minus[i] -= eps
        grad[i] = (f(x_plus) - f(x_minus)) / (2 * eps)
    return grad

test_points = [np.array([1.0, 2.0]), np.array([-1.0, 0.5]), np.array([3.0, -1.0])]

print("{:20s}  {:30s}  {:30s}  {}".format("x", "Analytical", "Numerical", "Max error"))
print("-" * 90)
for x0 in test_points:
    ag = grad_g(x0)
    ng = numerical_grad(g, x0)
    err = np.max(np.abs(ag - ng))
    print("{:20s}  {:30s}  {:30s}  {:.2e}".format(
        str(x0), str(ag.round(4)), str(ng.round(4)), err))

**What to notice:** the analytical and numerical gradients agree to about `1e-10` at
every test point. Central differences are accurate to `O(ε²)`, so with `ε = 1e-5` you
get ~10 correct digits — enough to trust a hand-derived gradient, which is exactly what
"gradient checking" does before training a network.

## 2. The library way — automatic differentiation

In real ML you never derive gradients by hand or approximate them: **autodiff** applies
the chain rule through your code and returns *exact* derivatives at machine precision.
`jax.grad(g)` returns a new function that computes `∇g`. (PyTorch's `autograd` and
TensorFlow's `GradientTape` do the same thing; `jax.grad` is the most compact to show.)
The cell confirms it matches both gradients from §1.

In [ ]:
import jax, jax.numpy as jnp
from jax import grad

def g_jax(x):                       # same function, written for jax
    return x[0]**2 + 2*x[1]**2 + x[0]*x[1]

autodiff_grad = grad(g_jax)         # ∇g, computed exactly by the chain rule

x0 = jnp.array([1.0, 2.0])
ad = np.array(autodiff_grad(x0))
print('autodiff  ∇g(1,2) =', ad)
print('analytical∇g(1,2) =', grad_g(np.array([1.0, 2.0])))
print('numerical ∇g(1,2) =', numerical_grad(g, np.array([1.0, 2.0])).round(6))

assert np.allclose(ad, grad_g(np.array([1.0, 2.0]))), "autodiff must match the analytical gradient"
print('\nautodiff == analytical ✓  (exact, no step-size to tune)')

**What to notice:** all three gradients are `[4, 9]`. But autodiff got there *exactly*
and with **no `ε` to tune** — it differentiates the operations in your code directly.
That's why frameworks train million-parameter networks without anyone ever writing a
gradient by hand: `loss.backward()` is this, scaled up.

## 3. Seeing it — tangent, descent, and the sigmoid derivative

First, the geometric meaning: the derivative `f'(x₀)` is the **slope of the tangent
line** touching the curve at `x₀`. The figure draws `f(x) = x³ − 2x² + x` with tangents
at three points.

In [ ]:
def f(x): return x**3 - 2*x**2 + x
def df(x): return 3*x**2 - 4*x + 1  # analytical derivative

x = np.linspace(-0.5, 2.5, 400)
tangent_points = [0.2, 1.0, 2.0]

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(x, f(x), color='#6366f1', lw=2.5, label='f(x) = x³ - 2x² + x')

colors = ['#f97316', '#2dd4bf', '#f59e0b']
for x0, color in zip(tangent_points, colors):
    slope = df(x0)
    tangent = f(x0) + slope * (x - x0)
    # Only draw tangent in a window
    mask = abs(x - x0) < 0.4
    ax.plot(x[mask], tangent[mask], '--', color=color, lw=2,
            label=f"f'({x0}) = {slope:.2f}")
    ax.scatter([x0], [f(x0)], color=color, s=60, zorder=5)

ax.set_xlabel('x'); ax.set_ylabel('f(x)')
ax.grid(True, alpha=0.2); ax.legend()
ax.set_title('Derivative = slope of tangent line', pad=12)
plt.tight_layout(); plt.show()

**What to notice:** each dashed tangent matches the curve's steepness at its point —
positive slope where `f` rises, negative where it falls, and near-zero at the turning
points. Those zero-slope points are exactly where optimization wants to stop.

Now the gradient put to work: **gradient descent** repeatedly steps against `∇loss` to
roll downhill toward the minimum of a 2-D bowl.

In [ ]:
def loss(w): return (w[0] - 3)**2 + 2*(w[1] + 1)**2   # minimum at (3, -1)
def grad_loss(w): return np.array([2*(w[0]-3), 4*(w[1]+1)])

# Run gradient descent
w = np.array([-2.0, 3.0])   # start far from minimum
lr = 0.1
path = [w.copy()]

for _ in range(30):
    w = w - lr * grad_loss(w)
    path.append(w.copy())

path = np.array(path)

# Plot
xx, yy = np.meshgrid(np.linspace(-3, 5, 200), np.linspace(-3, 5, 200))
Z = (xx - 3)**2 + 2*(yy + 1)**2

fig, ax = plt.subplots(figsize=(8, 6))
cs = ax.contourf(xx, yy, Z, levels=20, cmap='twilight', alpha=0.7)
ax.contour(xx, yy, Z, levels=20, colors='white', alpha=0.2, linewidths=0.5)
plt.colorbar(cs, ax=ax, label='Loss')

ax.plot(path[:, 0], path[:, 1], 'o-', color='#f97316', ms=5, lw=2, label='GD path')
ax.scatter([path[0,0]], [path[0,1]], color='#2dd4bf', s=100, zorder=6, label='Start')
ax.scatter([3], [-1], marker='*', color='#f59e0b', s=200, zorder=6, label='Minimum')

ax.set_xlabel('w₁'); ax.set_ylabel('w₂')
ax.set_title('Gradient descent on f(w₁,w₂) = (w₁−3)² + 2(w₂+1)²', pad=12)
ax.legend()
plt.tight_layout(); plt.show()

print(f'Final w: {path[-1].round(4)}  (true minimum: [3, -1])')
print(f'Final loss: {loss(path[-1]):.6f}')

**What to notice:** the path always crosses the contour lines at right angles — the
negative gradient is perpendicular to the level sets — and the steps *shrink* as the
slope flattens near the minimum, landing on `(3, −1)`. This exact loop, with `∇loss`
supplied by autodiff, is how every neural network trains.

## The sigmoid derivative, derived and verified

Using the chain rule on $\sigma(x) = (1 + e^{-x})^{-1}$:

$$\sigma'(x) = \frac{e^{-x}}{(1+e^{-x})^2} = \frac{1}{1+e^{-x}}\cdot\frac{e^{-x}}{1+e^{-x}} = \sigma(x)\,(1-\sigma(x)).$$

So the gradient is a cheap function of the forward output. Below we confirm the closed form matches a finite-difference derivative, and see why a saturated unit ($|x|$ large) has a near-zero gradient.

In [ ]:
def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))

def sigmoid_deriv_closed(x):
    s = sigmoid(x)
    return s * (1.0 - s)            # the σ(1-σ) shortcut

# Confirm the closed form matches a finite-difference derivative everywhere.
xs = np.linspace(-6, 6, 25)
eps = 1e-6
numeric = (sigmoid(xs + eps) - sigmoid(xs - eps)) / (2 * eps)
closed  = sigmoid_deriv_closed(xs)
print('max |closed-form - numerical| =', np.max(np.abs(closed - numeric)))

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(xs, sigmoid(xs), color='#6366f1', lw=2.5, label='σ(x)')
ax.plot(xs, closed, color='#f97316', lw=2.5, label="σ'(x) = σ(1-σ)")
ax.scatter(xs, numeric, color='#2dd4bf', s=12, zorder=5, label='σ′ numerical')
ax.axhline(0, color='#30344a')
ax.set_title("Sigmoid derivative peaks at x=0 (0.25) and vanishes in the tails", pad=10)
ax.set_xlabel('x'); ax.grid(True, alpha=0.2); ax.legend()
plt.tight_layout(); plt.show()
print(f"σ'(0) = {sigmoid_deriv_closed(np.array([0.0]))[0]:.3f}  (max slope)")
print(f"σ'(6) = {sigmoid_deriv_closed(np.array([6.0]))[0]:.5f}  (saturated -> gradient ~0)")

**What to notice:** the closed form `σ(1−σ)` matches the finite-difference derivative
everywhere, peaks at `0.25` when `x = 0`, and **vanishes** in the tails (`σ'(6) ≈ 0.002`).
That tail behavior is the seed of the *vanishing gradient* problem: stack many saturated
sigmoids and the backprop signal decays toward zero.

## 4. Gotchas & numerical traps

- **Finite-difference step size is a trade-off.** Too *large* an `ε` and the
  approximation is crude (truncation error); too *small* and floating-point
  **cancellation** wrecks it (subtracting nearly-equal numbers). There's a sweet spot
  around `1e-5` for central differences.
- **Saturation kills gradients.** A unit deep in a sigmoid/tanh tail has slope ≈ 0, so
  no learning signal flows back — the reason ReLU and normalization exist.
- **Learning rate can diverge.** Step against the gradient too far and you overshoot
  *further* each step (Exercise 2 makes this explode).
- **Kinks aren't differentiable.** ReLU has no derivative at 0; frameworks just pick a
  **subgradient** (0 or 1) and move on.

In [ ]:
# 1) The finite-difference U-curve: error vs step size h
true = np.exp(1.0)                      # d/dx eˣ at x=1 is e
print('   h        central-diff error')
for h in [1e-1, 1e-3, 1e-5, 1e-7, 1e-9, 1e-11]:
    approx = (np.exp(1 + h) - np.exp(1 - h)) / (2 * h)
    print(f'  {h:.0e}     {abs(approx - true):.2e}')

# 2) A non-differentiable kink: ReLU at 0
relu = lambda x: np.maximum(0.0, x)
h = 1e-5
print('\ncentral diff of ReLU at 0 =', (relu(h) - relu(-h)) / (2 * h),
      '(true derivative is undefined; frameworks use a subgradient of 0 or 1)')

**What to notice:** the error **falls then rises** — best near `h = 1e-5`, then
round-off takes over and it gets *worse* for smaller steps (`1e-11` is far off). And the
central difference of ReLU at 0 returns `0.5`, splitting the undefined kink — a reminder
that "the derivative" isn't always well defined, even when the code happily returns a
number.

## Key takeaways

- A **derivative** is a slope / sensitivity; the **gradient** stacks partials into the
  steepest-ascent vector, and training descends its negative.
- **Central differences** approximate derivatives to `O(ε²)` — great for *checking*
  gradients, but with a step-size sweet spot (truncation vs round-off).
- **Autodiff** (`jax.grad`, PyTorch `autograd`) gives *exact* gradients with no step to
  tune — the engine under `loss.backward()`.
- Watch for **saturation** (vanishing gradients), **too-large learning rates**
  (divergence), and **non-differentiable kinks** (subgradients).

**Next:** [Chain Rule & Backpropagation](https://ml-viz-ruby.vercel.app/courses/calculus-for-ml/02-chain-rule-and-backpropagation)
— how autodiff actually propagates these derivatives through a network.

---
## ✏️ Your turn

The cells below are **exercise scaffolds**: the concept is recapped, the code outline is set, and `# TODO(you)` marks what you fill in. Run the `assert` cell after each — it passes silently when your answer is right.

### Exercise 1 — Central differences

The derivative is a limit; a computer approximates it with a *small step*. The **central difference**

$$f'(x) \approx \frac{f(x+h) - f(x-h)}{2h}$$

is far more accurate than the one-sided version (error $O(h^2)$ vs $O(h)$) — it's exactly what you used above to verify analytical gradients, and what `scipy` / gradient-checking code does. Implement it.

In [ ]:
def numerical_derivative(f, x, h=1e-5):
    """Central-difference approximation of f'(x)."""
    # TODO(you): evaluate f a half-step h to each side and divide by the gap 2h
    return ...

In [ ]:
# Checks — run me
assert abs(numerical_derivative(lambda x: x ** 2, 3.0) - 6) < 1e-6, "d/dx x² at 3 is 6"
assert abs(numerical_derivative(np.sin, 0.0) - 1) < 1e-6, "d/dx sin at 0 is cos(0) = 1"
assert abs(numerical_derivative(np.exp, 1.0) - np.e) < 1e-5, "d/dx eˣ at 1 is e"
assert abs(numerical_derivative(lambda x: 5.0, 2.0)) < 1e-9, "constants have zero slope"
print("✅ Exercise 1 passed")

<details>
<summary>💡 Show solution</summary>

```python
def numerical_derivative(f, x, h=1e-5):
    return (f(x + h) - f(x - h)) / (2 * h)
```

</details>

### Exercise 2 — Gradient descent in 1D

Gradient descent repeats one move: step *against* the derivative, scaled by the learning rate:

$$x \leftarrow x - \eta \, f'(x)$$

Implement the loop and minimize $f(x) = x^2 - 4x + 5$ (so $f'(x) = 2x - 4$, minimum at $x = 2$). The last check shows the dark side: with $\eta > 1$ on this function, every step overshoots *further* than the last, and the iterates explode.

In [ ]:
def gradient_descent(grad, x0, lr=0.1, steps=100):
    """Run `steps` gradient-descent updates from x0; return the final x."""
    x = x0
    for _ in range(steps):
        # TODO(you): the update rule x <- x - lr * grad(x)
        x = ...
    return x

In [ ]:
# Checks — run me
grad = lambda x: 2 * x - 4   # f(x) = x² - 4x + 5

assert abs(gradient_descent(grad, x0=10.0) - 2) < 1e-6, "minimum of x² - 4x + 5 is at x = 2"
assert abs(gradient_descent(grad, x0=-7.0) - 2) < 1e-6, "converges from the other side too"
assert abs(gradient_descent(grad, x0=10.0, lr=1.1, steps=50) - 2) > 1e3, \
    "lr > 1 overshoots further every step and diverges"
print("✅ Exercise 2 passed")

<details>
<summary>💡 Show solution</summary>

```python
def gradient_descent(grad, x0, lr=0.1, steps=100):
    x = x0
    for _ in range(steps):
        x = x - lr * grad(x)
    return x
```

</details>

### Exercise 3 — Extra practice: derivative of a polynomial ([DML #116](https://github.com/Open-Deep-ML/DML-OpenProblem/tree/main/questions/116_derivative-of-a-polynomial))

DML's problem asks for the derivative of a single polynomial term $c \cdot x^n$ at a point $x$. By the power rule,

$$\frac{d}{dx}\big(c\,x^n\big) = c\,n\,x^{n-1}, \qquad \text{except } n = 0 \Rightarrow \frac{d}{dx}(c) = 0$$

(the $n=0$ special case matters because $x^0=1$ is a constant, and the naive formula $n\,x^{n-1}$ would try to evaluate $x^{-1}$).

A polynomial is a *sum* of such terms, so its derivative is the term-by-term sum too. We'll represent a polynomial as a coefficient list ordered **low-to-high degree** — `coeffs[i]` is the coefficient of $x^i$, so `coeffs = [c0, c1, ..., ck]` means $p(x) = c_0 + c_1 x + \dots + c_k x^k$ (this is the same low-to-high convention as `numpy.polynomial.Polynomial`, *not* `np.poly1d`'s reversed one). Implement `poly_derivative` by applying the power rule to every term, then we'll cross-check the result against the central-difference machinery from Exercise 1 and the sigmoid check above.

In [ ]:
def poly_term_derivative(c, x, n):
    """DML #116: derivative of a single term c * x**n at point x (power rule)."""
    if n == 0:
        return 0.0
    return c * n * x ** (n - 1)


def polyval(coeffs, x):
    """Evaluate p(x) given low-to-high coefficients: coeffs[i] is the x**i term."""
    return sum(c * x ** i for i, c in enumerate(coeffs))


def poly_derivative(coeffs):
    """Return the coefficient list of p'(x) (one shorter than `coeffs`).

    Apply the power rule (DML #116) term-by-term: the x**i term with
    coefficient coeffs[i] contributes i * coeffs[i] to position (i - 1)
    of the derivative.
    """
    # TODO(you): build the derivative's coefficient list from coeffs[1:]
    return ...

In [ ]:
# Checks — run me
assert poly_derivative([5.0]) == [], "constant polynomial p(x)=5 -> derivative is the zero polynomial"
assert poly_derivative([3.0, 2.0]) == [2.0], "linear p(x)=3+2x -> derivative is the constant 2"
assert poly_derivative([1.0, 0.0, 0.0, 0.0, 5.0]) == [0.0, 0.0, 0.0, 20.0], \
    "high-degree p(x)=1+5x^4 -> derivative is 20x^3"

# Cross-check the symbolic derivative against central differences -- the same
# numerical-vs-analytical comparison as Exercise 1 and the sigmoid check above.
coeffs = [2.0, -3.0, 0.0, 4.0]          # p(x) = 2 - 3x + 4x^3
deriv_coeffs = poly_derivative(coeffs)
h = 1e-5
for x0 in [-2.0, 0.0, 1.5, 3.0]:
    numeric = (polyval(coeffs, x0 + h) - polyval(coeffs, x0 - h)) / (2 * h)
    symbolic = polyval(deriv_coeffs, x0)
    assert abs(numeric - symbolic) < 1e-4, f"numerical and symbolic derivatives disagree at x={x0}"

print("✅ Exercise 3 passed")

<details>
<summary>💡 Show solution</summary>

```python
def poly_derivative(coeffs):
    return [i * coeffs[i] for i in range(1, len(coeffs))]
```

</details>